# RAG with LangChain & Pinecone (Local Embeddings Edition)

Retrieval-Augmented Generation using:
- **LLM:** GPT-4o-mini (via GitHub Models - FREE)
- **Embeddings:** all-MiniLM-L6-v2 (Local - FREE)
- **Vector Store:** Pinecone

---

## 1. Setup & Dependencies

In [1]:
%pip install -qU langchain langchain-openai langchain-pinecone langchain-community langchain-text-splitters pinecone beautifulsoup4 langchain-huggingface sentence-transformers

Note: you may need to restart the kernel to use updated packages.


In [2]:
import getpass
import os

if not os.environ.get("GITHUB_TOKEN"):
    os.environ["GITHUB_TOKEN"] = getpass.getpass("Enter your Fine-grained GitHub Token (github_pat_...): ")

if not os.environ.get("PINECONE_API_KEY"):
    os.environ["PINECONE_API_KEY"] = getpass.getpass("Enter your Pinecone API key: ")

print("Tokens configured. OpenAI key NOT required.")

Tokens configured. OpenAI key NOT required.


## 2. Initialize Components

In [3]:
from langchain_openai import ChatOpenAI
from langchain_huggingface import HuggingFaceEmbeddings

# LLM via GitHub Models
model = ChatOpenAI(
    model="gpt-4o-mini",
    api_key=os.environ["GITHUB_TOKEN"],
    base_url="https://models.inference.ai.azure.com"
)

# Local Embeddings (Running on your machine)
embeddings = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")

print(f"LLM: GitHub Models ({model.model_name})")
print(f"Embeddings: Local HuggingFace (all-MiniLM-L6-v2)")

/mnt/data/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


LLM: GitHub Models (gpt-4o-mini)
Embeddings: Local HuggingFace (all-MiniLM-L6-v2)


## 3. Pinecone Vector Store Configuration

In [4]:
from pinecone import Pinecone, ServerlessSpec
from langchain_pinecone import PineconeVectorStore

pc = Pinecone(api_key=os.environ.get("PINECONE_API_KEY"))

index_name = "arep-lab04-rag-local"

if not pc.has_index(index_name):
    pc.create_index(
        name=index_name,
        dimension=384, # all-MiniLM-L6-v2 dimension
        metric="cosine",
        spec=ServerlessSpec(cloud="aws", region="us-east-1"),
    )
    print(f"Created index: {index_name}")
else:
    print(f"Index already exists: {index_name}")

index = pc.Index(index_name)
vector_store = PineconeVectorStore(index=index, embedding=embeddings)

print(f"Vector store ready.")

Index already exists: arep-lab04-rag-local
Vector store ready.


## 4. Indexing Phase

### 4.1 Load Documents

In [5]:
import bs4
from langchain_community.document_loaders import WebBaseLoader

bs4_strainer = bs4.SoupStrainer(class_=("post-title", "post-header", "post-content"))

loader = WebBaseLoader(
    web_paths=("https://lilianweng.github.io/posts/2023-06-23-agent/",),
    bs_kwargs={"parse_only": bs4_strainer},
)

docs = loader.load()

print(f"Loaded {len(docs)} document(s)")
print(f"Total characters: {len(docs[0].page_content)}")

USER_AGENT environment variable not set, consider setting it to identify your requests.


Loaded 1 document(s)
Total characters: 43047


### 4.2 Split Documents

In [6]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=200,
    add_start_index=True,
)

all_splits = text_splitter.split_documents(docs)

print(f"Split into {len(all_splits)} chunks")

Split into 63 chunks


### 4.3 Store in Pinecone

In [7]:
document_ids = vector_store.add_documents(documents=all_splits)

print(f"Indexed {len(document_ids)} documents in Pinecone")

Indexed 63 documents in Pinecone


## 5. Retrieval & Generation Phase

### 5.1 Similarity Search

In [8]:
results = vector_store.similarity_search("What is task decomposition?", k=3)

for i, doc in enumerate(results):
    print(f"--- Result {i+1} ---")
    print(f"Content: {doc.page_content[:200]}...")

--- Result 1 ---
Content: Task decomposition can be done (1) by LLM with simple prompting like "Steps for XYZ.\n1.", "What are the subgoals for achieving XYZ?", (2) by using task-specific instructions; e.g. "Write a story outl...
--- Result 2 ---
Content: Task decomposition can be done (1) by LLM with simple prompting like "Steps for XYZ.\n1.", "What are the subgoals for achieving XYZ?", (2) by using task-specific instructions; e.g. "Write a story outl...
--- Result 3 ---
Content: Task decomposition can be done (1) by LLM with simple prompting like "Steps for XYZ.\n1.", "What are the subgoals for achieving XYZ?", (2) by using task-specific instructions; e.g. "Write a story outl...


### 5.2 RAG Chain

In [9]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough

# Retriever from vector store
retriever = vector_store.as_retriever(search_kwargs={"k": 3})

# Format retrieved documents into a single context string
def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)

# RAG prompt
rag_prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a helpful assistant. Answer the question based ONLY on the following context:\n\n{context}"),
    ("human", "{question}"),
])

# Build RAG chain using LCEL (LangChain Expression Language)
rag_chain = (
    {"context": retriever | format_docs, "question": RunnablePassthrough()}
    | rag_prompt
    | model
    | StrOutputParser()
)

print("RAG chain ready.")

RAG chain ready.


### 5.3 Query Demo

In [10]:
query = "What is task decomposition?"
response = rag_chain.invoke(query)
print(response)

Task decomposition is the process of breaking down a larger task into smaller, manageable sub-tasks or steps. This can be done in several ways, including:

1. Using a language model (LLM) with simple prompting, such as asking for steps or subgoals for achieving a specific task.
2. Providing task-specific instructions, like asking for a story outline when writing a novel.
3. Involving human inputs to guide the breakdown of the task.

Additionally, there is an approach known as LLM+P, which involves using an external classical planner for long-horizon planning. This approach utilizes the Planning Domain Definition Language (PDDL) to describe the planning problem, where the LLM translates the problem into PDDL, requests a planner to generate a PDDL plan, and then translates the plan back into natural language.


## 6. Cleanup (Optional)

In [11]:
# pc.delete_index(index_name)
# print(f"Deleted index: {index_name}")